In [40]:
import pandas as pd
import numpy as np

In [41]:
df = pd.read_csv("../data/dataset_mood_smartphone.csv")
df.head()

,Unnamed: 0,id,time,variable,value
0,1,AS14.01,2014-02-26 13:00:00.000,mood,6.0
1,2,AS14.01,2014-02-26 15:00:00.000,mood,6.0
2,3,AS14.01,2014-02-26 18:00:00.000,mood,6.0
3,4,AS14.01,2014-02-26 21:00:00.000,mood,7.0
4,5,AS14.01,2014-02-27 09:00:00.000,mood,6.0


In [42]:
df = df.drop("Unnamed: 0", axis=1)
df["time"] = pd.to_datetime(df["time"], errors="coerce", format="mixed")
df["date"] = df["time"].dt.date

print(f"Dataset shape: {df.shape}")
print(f"Unique users: {df['id'].nunique()}")
print(f"Variables: {df['variable'].nunique()}")
print(f"Total missing values in 'value': {df['value'].isnull().sum()}")

df.head()

Dataset shape: (376912, 5)
Unique users: 27
Variables: 19
Total missing values in 'value': 202


,id,time,variable,value,date
0,AS14.01,2014-02-26 13:00:00,mood,6.0,2014-02-26
1,AS14.01,2014-02-26 15:00:00,mood,6.0,2014-02-26
2,AS14.01,2014-02-26 18:00:00,mood,6.0,2014-02-26
3,AS14.01,2014-02-26 21:00:00,mood,7.0,2014-02-26
4,AS14.01,2014-02-27 09:00:00,mood,6.0,2014-02-27


# Outliers Removal

In [43]:
# Range validation
valid_ranges = {
    "mood": (1, 10),
    "circumplex.arousal": (-2, 2),
    "circumplex.valence": (-2, 2),
    "activity": (0, 1),
    "call": (0, 1),
    "sms": (0, 1),
}

duration_vars = ["screen"] + [v for v in df["variable"].unique() if v.startswith("appCat")]
total_before = df["value"].notna().sum()

# Flag out-of-range values as NaN
for var, (lo, hi) in valid_ranges.items():
    mask = (df["variable"] == var) & ((df["value"] < lo) | (df["value"] > hi))
    df.loc[mask, "value"] = np.nan

# Duration variables cannot be negative
for var in duration_vars:
    mask = (df["variable"] == var) & (df["value"] < 0)
    df.loc[mask, "value"] = np.nan

range_removed = total_before - df["value"].notna().sum()
print(f"Values removed by range validation: {range_removed}")


def zscore_outliers(group, threshold=3):
    """Flag values more than `threshold` standard deviations from the group mean.
    Z-scores are computed within each (user, variable) group.

    Args:
        group (pd.Series): Values for one (user, variable) group.
        threshold (float): Z-score cutoff used to mark outliers.

    Returns:
        pd.Series: Original values with outliers replaced by NaN.
    """
    mean = group.mean()
    std = group.std()
    if std == 0 or pd.isna(std):
        return group  # no variation, nothing to flag
    z_scores = (group - mean).abs() / std
    return group.where(z_scores <= threshold)


before_zscore = df["value"].notna().sum()

# Apply per user per variable
df["value"] = df.groupby(["id", "variable"])["value"].transform(zscore_outliers, threshold=3)

zscore_removed = before_zscore - df["value"].notna().sum()
total_removed = range_removed + zscore_removed

print(f"Values removed by z-score (3 std): {zscore_removed}")
print(f"Total values removed: {total_removed} ({total_removed / total_before * 100:.2f}%)")
print(f"\nTotal missing values now: {df['value'].isnull().sum()}")

Values removed by range validation: 4
Values removed by z-score (3 std): 4076
Total values removed: 4080 (1.08%)

Total missing values now: 4282


# Missing Value Imputation

## Gap Analysis

In [44]:
def analyze_gaps(group):
    """Count consecutive NaN sequences and their lengths.

    Args:
       group (pd.DataFrame): Data for one (user, variable) group.

    Returns:
       pd.Series: Statistics about missing value gaps."""
    is_null = group["value"].isnull()
    blocks = is_null.ne(is_null.shift()).cumsum()
    null_blocks = blocks[is_null]
    if null_blocks.empty:
        return pd.Series({"n_missing": 0, "n_gaps": 0, "max_gap": 0, "mean_gap": 0})
    gap_lengths = null_blocks.groupby(null_blocks).size()
    return pd.Series(
        {
            "n_missing": int(is_null.sum()),
            "n_gaps": len(gap_lengths),
            "max_gap": int(gap_lengths.max()),
            "mean_gap": round(gap_lengths.mean(), 1),
        }
    )


gap_stats = df.groupby(["id", "variable"]).apply(analyze_gaps).reset_index()
gap_stats = gap_stats[gap_stats["n_missing"] > 0].sort_values("max_gap", ascending=False)

print(f"Total (user, variable) groups with missing data: {len(gap_stats)}")
print(f"\nGap length distribution:")
print(f"  Short (≤3):   {(gap_stats['max_gap'] <= 3).sum()} groups")
print(f"  Medium (4-7): {((gap_stats['max_gap'] > 3) & (gap_stats['max_gap'] <= 7)).sum()} groups")
print(f"  Long (>7):    {(gap_stats['max_gap'] > 7).sum()} groups")
print(f"\nTop 10 longest gaps:")
print(gap_stats[["id", "variable", "n_missing", "max_gap"]].head(10).to_string(index=False))

Total (user, variable) groups with missing data: 310

Gap length distribution:
  Short (≤3):   297 groups
  Medium (4-7): 13 groups
  Long (>7):    0 groups

Top 10 longest gaps:
     id             variable  n_missing  max_gap
AS14.26             activity       21.0      6.0
AS14.14             activity       28.0      5.0
AS14.12             activity       14.0      5.0
AS14.31             activity       17.0      4.0
AS14.33             activity       31.0      4.0
AS14.05             activity       27.0      4.0
AS14.24   circumplex.valence       32.0      4.0
AS14.01   circumplex.valence       20.0      4.0
AS14.08             activity       34.0      4.0
AS14.01 appCat.communication       73.0      4.0


## Method 1: Temporal Interpolation (linear, limit=3)

In [45]:
df_temporal = df.copy()

missing_before = df_temporal["value"].isnull().sum()

df_temporal["value"] = df_temporal.groupby(["id", "variable"])["value"].transform(
    lambda x: x.interpolate(method="linear", limit=3)
)

missing_after_temporal = df_temporal["value"].isnull().sum()
print(f"Missing before imputation: {missing_before}")
print(f"Missing after temporal interpolation: {missing_after_temporal}")
print(f"Values imputed: {missing_before - missing_after_temporal}")

Missing before imputation: 4282
Missing after temporal interpolation: 29
Values imputed: 4253


## Method 2: Person-level Mean Imputation

In [46]:
df_person_mean = df.copy()

df_person_mean["value"] = df_person_mean.groupby(["id", "variable"])["value"].transform(
    lambda x: x.fillna(x.mean())
)

missing_after_person = df_person_mean["value"].isnull().sum()
print(f"Missing before imputation: {missing_before}")
print(f"Missing after person-mean imputation: {missing_after_person}")
print(f"Values imputed: {missing_before - missing_after_person}")

Missing before imputation: 4282
Missing after person-mean imputation: 0
Values imputed: 4282


## Comparison: Temporal vs Person-level Mean
We artificially mask 10% of known values and measure how well each method recovers them using MAE (Mean Absolute Error).

In [47]:
np.random.seed(42)

df_eval = df.copy()
known_mask = df_eval["value"].notna()
known_indices = df_eval[known_mask].index

# Randomly mask 10% of known values
mask_indices = np.random.choice(known_indices, size=int(0.1 * len(known_indices)), replace=False)
true_values = df_eval.loc[mask_indices, "value"].copy()
df_eval.loc[mask_indices, "value"] = np.nan

# Method 1: Temporal interpolation
df_eval_temporal = df_eval.copy()
df_eval_temporal["value"] = df_eval_temporal.groupby(["id", "variable"])["value"].transform(
    lambda x: x.interpolate(method="linear", limit=3)
)
pred_temporal = df_eval_temporal.loc[mask_indices, "value"]

# Method 2: Person-level mean
df_eval_person = df_eval.copy()
df_eval_person["value"] = df_eval_person.groupby(["id", "variable"])["value"].transform(
    lambda x: x.fillna(x.mean())
)
pred_person = df_eval_person.loc[mask_indices, "value"]

# Compute MAE for both methods
valid_temporal = pred_temporal.notna() & true_values.notna()
valid_person = pred_person.notna() & true_values.notna()

mae_temporal = (pred_temporal[valid_temporal] - true_values[valid_temporal]).abs().mean()
mae_person = (pred_person[valid_person] - true_values[valid_person]).abs().mean()

print(f"MAE Temporal interpolation: {mae_temporal:.4f}")
print(f"MAE Person-level mean:      {mae_person:.4f}")

MAE Temporal interpolation: 31.8517
MAE Person-level mean:      29.4557


## Final Cleaned Dataset

In [48]:
# Person-level mean imputation
df_clean = df.copy()

df_clean["value"] = df_clean.groupby(["id", "variable"])["value"].transform(
    lambda x: x.fillna(x.mean())
)

# Drop rows where user has no data at all for that variable
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=["value"])
rows_dropped = rows_before - len(df_clean)

print(f"Missing before imputation: {missing_before}")
print(f"Missing after person-mean: {df_clean['value'].isnull().sum()}")
print(f"Rows dropped (no data):    {rows_dropped}")
print(f"\nFinal dataset shape: {df_clean.shape}")

df_clean.to_csv("../data/dataset_mood_smartphone_cleaned.csv", index=False)
print("\nSaved to ../data/dataset_mood_smartphone_cleaned.csv")

Missing before imputation: 4282
Missing after person-mean: 0
Rows dropped (no data):    0

Final dataset shape: (376912, 5)

Saved to ../data/dataset_mood_smartphone_cleaned.csv
